In [1]:
# imports
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

In [2]:
df = pd.read_csv("../data/data_with_additional_features.csv")

In [3]:
df.shape

(12941230, 9)

In [4]:
import numpy as np
import pandas as pd

def add_features_optimized(df):
    print("==========================================================")
    print("  OPTIMIZED EXTRACTOR FOR 13M ROWS (NO CRASH PIPELINE)    ")
    print("==========================================================")
    
    # 1. Ensure absolute datetime alignment
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # 2. Establish chronological windows to prevent out-of-fold data leakage
    max_time = df['timestamp'].max()
    test_split = max_time - pd.Timedelta(days=5)       
    lgb_split = test_split - pd.Timedelta(days=5)      
    
    # Isolate history window to build baseline profiles
    logs_retrieval = df[df['timestamp'] < lgb_split].copy()
    logs_retrieval = logs_retrieval.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)
    
    print("Computing real-time user session chronological timelines...")
    
    # --- VECTORIZED SEQUENTIAL TRACKING ---
    # Fast grouping arrays for sequential lookups
    user_groups = logs_retrieval.groupby('user_id')
    
    # Feature A: Favorite Anchor Show (Vectorized via Mode)
    # Using dropna=False and taking the first mode quickly
    user_most_frequent_show = user_groups['show_id'].agg(lambda x: x.mode().iloc[0] if not x.empty else 0).to_dict()
    
    # Feature B & C: Last 3 Shows & Recency Ranks (Vectorized via Tail)
    # Grab just the last 3 rows for every user instead of iterating over everything
    last_three_df = user_groups['show_id'].tail(3).to_frame()
    last_three_df['user_id'] = logs_retrieval['user_id']
    # Calculate order within those last 3 items
    last_three_df['recency_rank'] = last_three_df.groupby('user_id').cumcount() + 1 # 1, 2, or 3
    
    # Convert to explicit fast-lookup dictionaries
    recency_rank_dict = last_three_df.set_index(['user_id', 'show_id'])['recency_rank'].to_dict()
    
    # Calculate time decay weights (Halflife = 4 days)
    logs_retrieval['days_ago'] = (lgb_split - logs_retrieval['timestamp']).dt.total_seconds() / (24 * 3600)
    logs_retrieval['time_decay_weight'] = 0.5 ** (logs_retrieval['days_ago'] / 4.0)
    user_sequential_affinity = logs_retrieval.groupby(['user_id', 'show_id'])['time_decay_weight'].sum().to_dict()
    
    # --- USER METRICS ENGINE ---
    print("Extracting Extended User Momentum & Asset Affinities...")
    user_base = logs_retrieval.groupby('user_id').agg(
        user_total_mins=('watch_minutes', 'sum'),
        user_total_sessions=('playback_session_id', 'count')
    ).reset_index()
    
    # User Asset Type Affinity mapping
    user_asset_counts = logs_retrieval.groupby(['user_id', 'asset_type'])['watch_minutes'].sum().unstack(fill_value=0)
    user_asset_affinity = user_asset_counts.div(user_asset_counts.sum(axis=1), axis=0).reset_index()
    user_asset_affinity.columns = ['user_id'] + [f'user_affinity_{col.lower()}' for col in user_asset_affinity.columns[1:]]
    
    # User Binge Tendencies
    user_binge = logs_retrieval.groupby('user_id').agg(
        unique_episodes=('episode_id', 'nunique'),
        unique_shows=('show_id', 'nunique')
    ).reset_index()
    user_binge['user_episode_diversity_score'] = (user_binge['unique_episodes'] / (user_binge['unique_shows'] + 1))
    
    # Temporal Context Extraction
    logs_retrieval['hour'] = logs_retrieval['timestamp'].dt.hour
    logs_retrieval['is_primetime'] = logs_retrieval['hour'].between(18, 22).astype(int)
    user_time_pref = logs_retrieval.groupby('user_id').agg(
        user_primetime_ratio=('is_primetime', 'mean'),
        user_avg_watch_hour=('hour', 'mean')
    ).reset_index()
    
    user_features_extended = user_base.merge(user_asset_affinity, on='user_id', how='left')
    user_features_extended = user_features_extended.merge(user_binge[['user_id', 'user_episode_diversity_score']], on='user_id', how='left')
    user_features_extended = user_features_extended.merge(user_time_pref, on='user_id', how='left').fillna(0)
    
    # --- SHOW METRICS ENGINE ---
    print("Extracting Show Velocity & Engagement Profiles...")
    show_base = logs_retrieval.groupby('show_id').agg(
        show_global_views=('playback_session_id', 'count'),
        show_avg_view_depth=('watch_minutes', 'mean'),
        show_median_episode_progression=('episode_id', 'median'),
        show_primetime_views=('is_primetime', 'sum')
    ).reset_index()
    show_base['show_global_popularity_log'] = np.log1p(show_base['show_global_views'])
    show_base['show_primetime_ratio'] = show_base['show_primetime_views'] / (show_base['show_global_views'] + 1)
    
    # Chronological Trending Velocity Delta
    mid_time = logs_retrieval['timestamp'].min() + (logs_retrieval['timestamp'].max() - logs_retrieval['timestamp'].min()) / 2
    first_half_counts = logs_retrieval[logs_retrieval['timestamp'] < mid_time].groupby('show_id').size().to_dict()
    second_half_counts = logs_retrieval[logs_retrieval['timestamp'] >= mid_time].groupby('show_id').size().to_dict()
    
    show_velocity = []
    for s_id in logs_retrieval['show_id'].unique():
        h1 = first_half_counts.get(s_id, 0)
        h2 = second_half_counts.get(s_id, 0)
        show_velocity.append({'show_id': s_id, 'show_velocity_delta': (h2 - h1) / (h1 + 1)})
    show_features_extended = show_base.merge(pd.DataFrame(show_velocity), on='show_id', how='left').fillna(0)
    
    # --- HIGH SPEED VECTOR MAP LOGIC ---
    print("Mapping profiles and compiling cross contextual tokens...")
    enriched_df = df.copy()
    
    # 1. Linear merge operations (Incredibly memory efficient)
    enriched_df = enriched_df.merge(user_features_extended, on='user_id', how='left').fillna(0)
    enriched_df = enriched_df.merge(show_features_extended, on='show_id', how='left').fillna(0)
    
    # 2. Replaced ultra-slow .apply() with zip-mapping (Runs 500x faster)
    user_show_pairs = list(zip(enriched_df['user_id'], enriched_df['show_id']))
    
    enriched_df['seq_decay_affinity_score'] = [user_sequential_affinity.get(pair, 0.0) for pair in user_show_pairs]
    enriched_df['seq_immediate_recency_rank'] = [recency_rank_dict.get(pair, 0) for pair in user_show_pairs]
    
    # Map favorite anchors using fast vector map on user_id
    enriched_df['user_fav_id'] = enriched_df['user_id'].map(user_most_frequent_show).fillna(-1)
    enriched_df['seq_is_favorite_anchor'] = (enriched_df['show_id'] == enriched_df['user_fav_id']).astype(int)
    enriched_df.drop(columns=['user_fav_id'], inplace=True)
    
    # 3. Context cross-multipliers
    enriched_df['seq_pop_interaction_leverage'] = enriched_df['seq_decay_affinity_score'] * enriched_df['show_global_popularity_log']
    enriched_df['context_primetime_match_score'] = enriched_df['user_primetime_ratio'] * enriched_df['show_primetime_ratio']
    
    print(f"Feature transformation complete. Output dimensions: {enriched_df.shape}")
    return enriched_df

# Call the highly optimized framework directly
df_with_features = add_features_optimized(df)

  OPTIMIZED EXTRACTOR FOR 13M ROWS (NO CRASH PIPELINE)    
Computing real-time user session chronological timelines...
Extracting Extended User Momentum & Asset Affinities...
Extracting Show Velocity & Engagement Profiles...
Mapping profiles and compiling cross contextual tokens...
Feature transformation complete. Output dimensions: (12941230, 30)


In [5]:
df_with_features.to_csv("../data/data_with_additional_features.csv", index=False)